# Advanced RAG with LangChain: Basic RAG vs Self-Query RAG

This notebook demonstrates two RAG approaches:

1. Basic RAG Flow - Simple similarity search
2. Self-Query RAG - Intelligent metadata filtering

Setup:
- Embeddings: BAAI/bge-small-en-v1.5 (lightweight)
- LLM: Llama 3.1 70B Instruct via OpenRouter
- Vector Store: Chroma

In [65]:
!pip install -q langchain tiktoken langchain-community langchain_chroma langchain-huggingface langchain-openai sentence-transformers lark python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup - Load API Keys and Initialize Models

In [66]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API Key: ")

print("API Key loaded successfully")

API Key loaded successfully


In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
# FIX: Import ChatOpenAI from langchain_openai instead of langchain_community
from langchain_openai import ChatOpenAI

# Initialize free local embeddings
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Initialize OpenRouter LLM using the dedicated OpenAI integration module
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-70b-instruct",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

print("Embeddings model loaded: BAAI/bge-small-en-v1.5")
print("LLM initialized: Llama 3.1 70B Instruct via OpenRouter")

Embeddings model loaded: BAAI/bge-small-en-v1.5
LLM initialized: Llama 3.1 70B Instruct via OpenRouter


In [68]:
import textwrap

def wrap_text(text, width=90):
    lines = text.split('\n')
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]
    return '\n'.join(wrapped_lines)

In [69]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="A hacker discovers reality is a simulation and leads a rebellion against the machines controlling it.",
        metadata={"year": 1999, "director": "Lana Wachowski, Lilly Wachowski", "rating": 8.7, "genre": "science fiction"},
    ),
    Document(
        page_content="A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        metadata={"year": 1994, "rating": 8.5, "genre": "animated"},
    ),
    Document(
        page_content="Batman faces off against the Joker, a criminal mastermind who plunges Gotham into chaos.",
        metadata={"year": 2008, "director": "Christopher Nolan", "rating": 9.0, "genre": "action"},
    ),
    Document(
        page_content="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"},
    ),
]

print(f"Loaded {len(docs)} movie documents with metadata")

Loaded 9 movie documents with metadata


## 2. Basic RAG Flow

### Pipeline Diagram

```mermaid
graph TD
    A[User Query] --> B[Embedding Model]
    B --> C[Vector Search]
    D[Document Store] --> C
    C --> E[Top-K Similar Chunks]
    E --> F[LLM with Context]
    F --> G[Answer]
    
    style A fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style B fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style C fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style D fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style E fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style F fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style G fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
```

When to use:
- Linear documents (PDFs, articles)
- No complex metadata filtering needed
- Simple similarity-based retrieval

Limitations:
- May retrieve similar but irrelevant chunks
- No understanding of metadata constraints
- Information loss due to lack of context awareness

In [70]:
from langchain_chroma import Chroma

vectorstore_basic = Chroma.from_documents(
    docs, 
    embedding,
    collection_name="basic_rag_movies",
    persist_directory="./chroma_db_basic"
)

print("Vector store created for Basic RAG")

Vector store created for Basic RAG


In [71]:
question1 = "Which 1994 animated movie has a rating of 8.5?"
question2 = "Which movie features Batman facing off against the Joker and who directed it?"
question3 = "What genre is the movie 'The Matrix' and who directed it?"

print("Testing Basic Similarity Search:\n")
print(f"Query: {question1}")
results = vectorstore_basic.similarity_search(question1, k=3)
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.page_content[:80]}...")
    print(f"   Metadata: {doc.metadata}")

Testing Basic Similarity Search:

Query: Which 1994 animated movie has a rating of 8.5?

1. Toys come alive and have a blast doing so...
   Metadata: {'year': 1995, 'genre': 'animated'}

2. Toys come alive and have a blast doing so...
   Metadata: {'genre': 'animated', 'year': 1995}

3. Toys come alive and have a blast doing so...
   Metadata: {'genre': 'animated', 'year': 1995}


In [72]:
# FIX: Import from langchain_core instead of langchain or langchain.schema
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever_basic = vectorstore_basic.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

chain_basic = (
    {"context": retriever_basic, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Basic RAG chain created")

Basic RAG chain created


In [73]:
print("=" * 90)
print("BASIC RAG DEMO")
print("=" * 90)

print(f"\nQuery 1: {question1}\n")
response1 = chain_basic.invoke(question1)
print(wrap_text(response1))

print("\n" + "-" * 90 + "\n")

print(f"Query 2: {question2}\n")
response2 = chain_basic.invoke(question2)
print(wrap_text(response2))

print("\n" + "-" * 90 + "\n")

print("Query 3: Tell me about movies with rating more than 7\n")
response3 = chain_basic.invoke("Tell me about movies with rating more than 7")
print(wrap_text(response3))

BASIC RAG DEMO

Query 1: Which 1994 animated movie has a rating of 8.5?

There is no information in the provided context about a 1994 animated movie with a rating
of 8.5. The context only mentions three documents with metadata from 1995, but no ratings
are mentioned.

------------------------------------------------------------------------------------------

Query 2: Which movie features Batman facing off against the Joker and who directed it?



The movie features Batman facing off against the Joker, and it was directed by Christopher
Nolan.

------------------------------------------------------------------------------------------

Query 3: Tell me about movies with rating more than 7

Based on the provided context, I can see that there are three documents that match the
criteria of having a rating more than 7. Here are the details about these movies:

* All three movies have a rating of 8.2.
* The director of all three movies is Christopher Nolan.
* The release year of all three mov

## 3. Self-Query RAG

### Theory

Self-Query RAG is an advanced retrieval technique that enhances traditional RAG by intelligently parsing user queries to extract both semantic content and metadata filters. Unlike Basic RAG, which relies solely on vector similarity, Self-Query RAG uses a Language Model to understand query intent and automatically construct structured database queries.

The process involves:

1. Query Analysis - LLM parses the natural language query to identify:
   - Semantic search terms (what to search for)
   - Metadata constraints (year, genre, rating, etc.)
   - Comparison operators (equals, greater than, less than)

2. Structured Query Generation - Converts parsed components into a structured format:
   - Query string for semantic search
   - Filter operations for metadata
   - Logical operators (AND, OR) for combining filters

3. Dual Filtering - Applies two retrieval mechanisms simultaneously:
   - Vector similarity search on content embeddings
   - Metadata filtering using structured query

4. Precise Retrieval - Returns documents that match BOTH:
   - Semantic relevance to the query
   - Metadata constraints specified

Benefits:
- Reduces false positives from pure similarity search
- Enables complex filtering without manual query construction
- Better handles queries with specific constraints
- Improves retrieval precision in large, diverse document collections

### Pipeline Diagram

```mermaid
graph TD
    A[User Query] --> B[LLM Query Constructor]
    B --> C[Structured Query]
    C --> D{Query Parts}
    D --> E[Semantic Filter]
    D --> F[Metadata Filter]
    E --> G[Vector Search]
    F --> G
    H[Document Store] --> G
    G --> I[Filtered Results]
    I --> J[LLM with Context]
    J --> K[Answer]
    
    style A fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style B fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style C fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style D fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style E fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style F fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style G fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style H fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style I fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style J fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
    style K fill:#FFFFFF,stroke:#000000,stroke-width:2px,color:#000000
```

When to use:
- Documents with rich metadata (year, category, author, ratings)
- Queries with specific constraints ("movies from 1990s", "rating > 8")
- Need to search within document subsets
- Complex filtering requirements

In [74]:
vectorstore_selfquery = Chroma.from_documents(
    docs,
    embedding,
    collection_name="selfquery_rag_movies",
    persist_directory="./chroma_db_selfquery"
)

print("Vector store created for Self-Query RAG")

Vector store created for Self-Query RAG


In [75]:
# FIX: Correct structural schema import for newer LangChain ecosystem
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float"
    ),
]

document_content_description = "Brief summary of a movie"

print("Metadata schema defined for self-querying")

Metadata schema defined for self-querying


In [76]:
from langchain_classic.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

from langchain_classic.chains.query_constructor.base import Comparator, Operator

prompt_selfquery = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
    allowed_comparators=[
        Comparator.EQ,   # equals
        Comparator.NE,   # not equals  
        Comparator.GT,   # greater than
        Comparator.GTE,  # greater than or equal
        Comparator.LT,   # less than
        Comparator.LTE   # less than or equal
    ],
    allowed_operators=[Operator.AND, Operator.OR]
)


output_parser = StructuredQueryOutputParser.from_components()

query_constructor = prompt_selfquery | llm | output_parser

print("Query constructor created")

Query constructor created


In [77]:
print("Testing Query Constructor:\n")
test_query = "What are some sci-fi movies from the 90's about simulations?"
print(f"Input: {test_query}\n")

structured_query = query_constructor.invoke({"query": test_query})
print(f"Structured Query:\n{structured_query}")

Testing Query Constructor:

Input: What are some sci-fi movies from the 90's about simulations?

Structured Query:
query='simulations' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='genre', value='science fiction'), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2000)]) limit=None


In [78]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

retriever_selfquery = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore_selfquery,
    structured_query_translator=ChromaTranslator(),
)

print("Self-Query Retriever created")

Self-Query Retriever created


In [79]:
print("Testing Self-Query Retrieval:\n")
test_query2 = "What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated"
print(f"Query: {test_query2}\n")

results = retriever_selfquery.invoke(test_query2)
print(f"Found {len(results)} result(s):\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   Metadata: {doc.metadata}\n")

Testing Self-Query Retrieval:

Query: What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated

Found 4 result(s):

1. Toys come alive and have a blast doing so
   Metadata: {'genre': 'animated', 'year': 1995}

2. Toys come alive and have a blast doing so
   Metadata: {'year': 1995, 'genre': 'animated'}

3. Toys come alive and have a blast doing so
   Metadata: {'genre': 'animated', 'year': 1995}

4. A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.
   Metadata: {'rating': 8.5, 'year': 1994, 'genre': 'animated'}



In [80]:
chain_selfquery = (
    {"context": retriever_selfquery, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Self-Query RAG chain created")

Self-Query RAG chain created


In [81]:
print("=" * 90)
print("SELF-QUERY RAG DEMO")
print("=" * 90)

queries = [
    "Tell me about movies with rating more than 8.5",
    "What are Christopher Nolan movies from after 2010?",
    "Which animated movie was released in 1994?",
    "What science fiction movies are in the database?"
]

for i, query in enumerate(queries, 1):
    print(f"\nQuery {i}: {query}\n")
    response = chain_selfquery.invoke(query)
    print(wrap_text(response))
    print("\n" + "-" * 90)

SELF-QUERY RAG DEMO

Query 1: Tell me about movies with rating more than 8.5

Based on the provided context, there are four movies with a rating of 8.6, which is more
than 8.5. Three of these movies are identical, with the same page content, and have the
following metadata:

* Director: Christopher Nolan
* Year: 2014
* Genre: science fiction

The fourth movie has the following metadata:

* Director: Satoshi Kon
* Year: 2006

Note that the genre for this fourth movie is not specified in the provided context.

------------------------------------------------------------------------------------------

Query 2: What are Christopher Nolan movies from after 2010?

Based on the provided context, the Christopher Nolan movies from after 2010 are:

* The movie described in Document(id='ec196817-186f-406e-b842-451b60fe8a50')
* The movie described in Document(id='576ba03f-e5db-4ce3-821b-eb1ff8ccf159')
* The movie described in Document(id='cccca034-c1cd-4d74-a68b-717d607741eb')

All three documents

## 4. Comparison: Basic RAG vs Self-Query RAG

| Feature | Basic RAG | Self-Query RAG |
|---------|-----------|----------------|
| Retrieval Method | Similarity search only | Similarity + Metadata filtering |
| Query Understanding | None | LLM parses intent |
| Metadata Awareness | No | Yes |
| Precision | Lower | Higher |
| Best For | Simple documents | Rich metadata documents |
| Complexity | Low | Medium |
| Cost | 1 LLM call | 2 LLM calls (parse + generate) |

When to use each:

Basic RAG:
- PDFs, articles, documentation
- No complex filtering needed
- Lower latency requirements

Self-Query RAG:
- Product catalogs, databases
- Queries with constraints (year, category, rating)
- Need precise filtering
- Multi-faceted searches